In [26]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
titles_emotions_path = PROJECT_ROOT / "data" / "processed" / "05_titles_emotion_scores.csv"

df = pd.read_csv(titles_emotions_path)

# select region, count(*) where dominant_emotion is not null
region_counts = df[df['dominant_emotion'].notnull()]['region'].value_counts()

print("Region Counts:")
print(region_counts)

display(df.head())

Region Counts:
region
Global       196
USA          196
Spain        192
Colombia     188
Japan        188
Singapore    183
Argentina    178
Taiwan       149
Name: count, dtype: int64


,rank,artist,title,region,spotify_uri,dominant_emotion,emotion_love,emotion_longing,emotion_joy,emotion_heartbreak,emotion_grief,emotion_despair,emotion_hope,emotion_lonely,emotion_sensual,emotion_anger
0,1,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,4nJJCRYru4QQakCiUA155f,sensual,0.0000,0.9144,0.1502,0.6234,0.6079,0.2024,0.1512,0.5652,0.9627,0.7008
1,2,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,3CBEVPwR3kUXDoTx1lqFUQ,lonely,0.8468,0.9427,0.4254,0.7883,0.7394,0.7509,0.7923,0.9984,0.9567,0.7817
2,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,2ZyrAym0sRLwt4PhGotHuI,sensual,0.2064,0.3978,0.3276,0.0000,0.1318,0.0000,0.1692,0.1429,0.9870,0.2859
3,4,Kris R.,GANAS,Colombia,4KE9Ne3hgh18B3Th4xcylg,longing,0.4590,0.9713,0.1610,0.6316,0.4115,0.2304,0.2900,0.7325,0.9208,0.5977
4,5,"W Sound, Beéle, Ovy On The Drums",La Plena - W Sound 05,Colombia,6iOndD4OFo7GkaDypWQIou,sensual,0.9199,0.9352,0.2911,0.0000,0.0000,0.0000,0.1598,0.0000,0.9919,0.0000


In [ ]:
# top regions of each emotion_score
for emotion in df.columns[6:]:
    print(f"Top regions for {emotion}:")
    top_regions = df.groupby('region')[emotion].mean().sort_values(ascending=False)
    print(top_regions)
    print("\n")

Top regions for emotion_longing:
region
Global       0.891261
USA          0.884761
Singapore    0.883022
Spain        0.859501
Argentina    0.857554
Taiwan       0.842012
Japan        0.830190
Colombia     0.822877
Name: emotion_longing, dtype: float64


Top regions for emotion_joy:
region
Taiwan       0.392873
Japan        0.340960
Singapore    0.326512
Global       0.303887
Colombia     0.273074
USA          0.258319
Spain        0.256359
Argentina    0.231156
Name: emotion_joy, dtype: float64


Top regions for emotion_heartbreak:
region
USA          0.655754
Singapore    0.616043
Global       0.614573
Argentina    0.608803
Taiwan       0.581352
Japan        0.550625
Spain        0.550483
Colombia     0.538506
Name: emotion_heartbreak, dtype: float64


Top regions for emotion_grief:
region
USA          0.624291
Global       0.587634
Singapore    0.579015
Taiwan       0.571679
Argentina    0.557770
Spain        0.539831
Colombia     0.505131
Japan        0.443174
Name: emotion_grief,

In [ ]:
#top emotions for each region
for region in df['region'].unique():
    print(f"Top emotions for {region}:")
    region_emotions = df[df['region'] == region].iloc[:, 6:].mean().sort_values(ascending=False)
    print(region_emotions)
    print("\n")

Top emotions for Colombia:
emotion_sensual       0.843853
emotion_longing       0.822877
emotion_heartbreak    0.538506
emotion_lonely        0.527501
emotion_anger         0.523456
emotion_grief         0.505131
emotion_despair       0.328009
emotion_hope          0.294929
emotion_joy           0.273074
dtype: float64


Top emotions for Global:
emotion_longing       0.891261
emotion_sensual       0.844262
emotion_heartbreak    0.614573
emotion_grief         0.587634
emotion_lonely        0.539445
emotion_anger         0.504415
emotion_hope          0.421500
emotion_despair       0.420155
emotion_joy           0.303887
dtype: float64


Top emotions for Taiwan:
emotion_longing       0.842012
emotion_sensual       0.806880
emotion_heartbreak    0.581352
emotion_anger         0.575243
emotion_grief         0.571679
emotion_lonely        0.520862
emotion_hope          0.468232
emotion_despair       0.457268
emotion_joy           0.392873
dtype: float64


Top emotions for USA:
emotion_longi

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

emotion_cols = df.columns[6:]
heatmap_data = df.groupby('region')[emotion_cols].mean()

fig_heatmap = px.imshow(
    heatmap_data.T,
    labels=dict(x="Region", y="Emotion", color="Mean Score"),
    x=heatmap_data.index,
    y=emotion_cols,
    color_continuous_scale="RdYlGn",
    aspect="auto",
    title="Emotion Intensity by Region (Heatmap)"
)
fig_heatmap.update_layout(height=500, width=1000)
fig_heatmap.show()

In [30]:
# Bubble Chart: Compare two emotions across regions
emotion_cols_list = list(emotion_cols)

# Use first two emotions for comparison
emotion1, emotion2 = emotion_cols_list[0], emotion_cols_list[1]

# Group by region and calculate mean scores + count of rows
bubble_means = df.groupby('region')[[emotion1, emotion2]].mean().reset_index()
bubble_counts = df.groupby('region').size().reset_index(name='song_count')
bubble_data = bubble_means.merge(bubble_counts, on='region', how='left')

fig_bubble = px.scatter(
    bubble_data,
    x=emotion1,
    y=emotion2,
    size='song_count',
    hover_data={'region': True, 'song_count': True},
    color='region',
    size_max=50,
    title=f"Bubble Chart: {emotion1} vs {emotion2} by Region",
    labels={emotion1: emotion1.replace('_', ' ').title(),
            emotion2: emotion2.replace('_', ' ').title()},
)
fig_bubble.update_layout(height=600, width=900)
fig_bubble.show()

In [31]:
# Bar Charts: Top regions for each emotion
emotion_cols_list = list(emotion_cols)
num_emotions = len(emotion_cols_list)

# Create subplots
from plotly.subplots import make_subplots

fig_bars = make_subplots(
    rows=(num_emotions + 2) // 3,
    cols=3,
    subplot_titles=emotion_cols_list,
    specs=[[{"type": "bar"} for _ in range(3)] for _ in range((num_emotions + 2) // 3)]
)

for idx, emotion in enumerate(emotion_cols_list):
    row = (idx // 3) + 1
    col = (idx % 3) + 1
    
    top_regions = df.groupby('region')[emotion].mean().sort_values(ascending=False).head(5)
    
    fig_bars.add_trace(
        go.Bar(x=top_regions.index, y=top_regions.values, name=emotion, showlegend=False),
        row=row, col=col
    )
    fig_bars.update_yaxes(title_text=emotion.replace('_', ' ').title(), row=row, col=col)

fig_bars.update_layout(height=300 * ((num_emotions + 2) // 3), width=1200, 
                       title_text="Top 5 Regions for Each Emotion (Bar Charts)")
fig_bars.show()

In [32]:
# Box Plots: Distribution of emotion scores by region
# Prepare data in long format for box plots
df_melted = df[['region'] + list(emotion_cols)].melt(id_vars='region', 
                                                       var_name='emotion', 
                                                       value_name='score')

fig_box = px.box(
    df_melted,
    x='emotion',
    y='score',
    color='emotion',
    facet_col='region',
    facet_col_wrap=3,
    title="Distribution of Emotion Scores by Region (Box Plots)",
    labels={'score': 'Score', 'emotion': 'Emotion'},
    height=800,
    width=1200
)
fig_box.show()

In [33]:
# Sunburst Chart: Hierarchical view (Region -> Emotion)
# Calculate mean scores by region and emotion
sunburst_data = df[['region'] + list(emotion_cols)].melt(
    id_vars='region',
    var_name='emotion',
    value_name='score'
)
sunburst_agg = sunburst_data.groupby(['region', 'emotion'], as_index=False)['score'].mean()

fig_sunburst = px.sunburst(
    sunburst_agg,
    path=['region', 'emotion'],
    values='score',
    color='score',
    color_continuous_scale='RdYlGn',
    title='Sunburst Chart: Emotions by Region',
    height=700,
    width=900
)
fig_sunburst.show()

In [ ]:
# ── Differential Heatmap: what makes each region distinctive ─────────────────
# Shows each region's mean emotion score MINUS the global mean.
# Positive = above global average; negative = below.
# This exposes regional character even when scores are generally high.

global_mean = df[emotion_cols].mean()
region_means = df.groupby('region')[emotion_cols].mean()
diff = region_means.subtract(global_mean)

# Rename columns for display
diff.columns = [c.replace('emotion_', '') for c in diff.columns]

fig_diff = px.imshow(
    diff.T,
    labels=dict(x="Region", y="Emotion", color="vs Global Mean"),
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="Regional Emotion Character — Deviation from Global Mean<br>"
          "<sup>Positive (red) = above average for that region; Negative (blue) = below</sup>",
)
fig_diff.update_layout(height=500, width=1000)
fig_diff.show()